<a href="https://colab.research.google.com/github/TrajanDS/SFT_Agent_POC/blob/main/data_gathering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
import requests
import pandas as pd
import time

In [76]:
CFPB_API_URL = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"

def fetch_cfpb_complaints(
    max_records=10_000,
    batch_size=1_000,
    date_received_min="2024-01-01",
    date_received_max="2026-01-01",
    has_narrative=True,
):
    """
    Purpose:
    Retrieves complaint data from the Consumer Financial Protection Bureau's
     API for Citizen Financial Group.

    ------------------------------------------------------------------------
    Parameters:
    max_records (int): Maximum number of records to fetch.
    batch_size (int): Number of records to fetch in each API call.

    ------------------------------------------------------------------------
    TODO:
      * Randomize time period being drawn from
      * Get set number of samples for each type
    """


    rows = []
    offset = 0
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; CFPB-DataFetcher/1.0; https://your-site-or-email)"
    }

    while len(rows) < max_records:
        current_batch = min(batch_size, max_records - len(rows))

        params = {
            "frm": offset,
            "size": current_batch,
            "no_aggs": "true",
            "company":"CITIZENS FINANCIAL GROUP, INC.",
            "date_received_min": date_received_min,
        }
        if date_received_max:
            params["date_received_max"] = date_received_max
        if has_narrative:
            params["has_narrative"] = "true"

        try:
            response = requests.get(CFPB_API_URL, params=params, headers=headers, timeout=60)
            response.raise_for_status()
            data = response.json()

            hits = data.get("hits", {}).get("hits", [])
            if not hits:
                print("No more results returned.")
                break

            # Extract the actual complaint data (_source)
            for hit in hits:
                if "_source" in hit:
                    rows.append(hit["_source"])
                else:
                    rows.append(hit)

            print(f"Fetched {len(hits)} records (total so far: {len(rows)})")

            offset += len(hits)

            # Be polite to the API
            if len(hits) < current_batch:
                break  # last page

            time.sleep(0.5)  # small delay to avoid rate-limiting

        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            break

    print(f"Completed: {len(rows)} records fetched.")
    return rows


# Call API
rows = fetch_cfpb_complaints()

# Form rows from API as dataframe
df = pd.DataFrame(rows)

Fetched 1000 records (total so far: 1000)
Fetched 1000 records (total so far: 2000)
Fetched 1000 records (total so far: 3000)
Fetched 1000 records (total so far: 4000)
Fetched 1000 records (total so far: 5000)
Fetched 1000 records (total so far: 6000)
Fetched 1000 records (total so far: 7000)
Fetched 1000 records (total so far: 8000)
Fetched 1000 records (total so far: 9000)
Fetched 1000 records (total so far: 10000)
Completed: 10000 records fetched.


In [77]:
df.company.value_counts()

,count
company,
"CITIZENS FINANCIAL GROUP, INC.",10000


In [78]:
df['product'].value_counts()

,count
product,
Checking or savings account,5470
Mortgage,1250
Credit reporting or other personal consumer reports,1170
Credit card,720
"Money transfer, virtual currency, or money service",390
Debt collection,330
"Payday loan, title loan, personal loan, or advance loan",270
Vehicle loan or lease,250
Student loan,140


In [65]:
tdf = pd.DataFrame(fetch_cfpb_complaints(
    max_records=1_000,
    batch_size=1_000,
    date_received_min="2024-01-01",
    date_received_max="2024-01-05",
    has_narrative=True,
)
)

Fetched 1000 records (total so far: 1000)
Completed: 1000 records fetched.


In [66]:
tdf.complaint_what_happened.values[0:4]

array(['I have requested for the credit agency to remove/delete a bankruptcy that is inaccurate. I also requested for the company to provide detailed information on the process that is used to verify bankruptcies but the agency failed to provide this information. This has been very difficult to fix. It has been over 30 days with multiple disputes without a solution/explanation. The company has not deleted/ removed this from my report causing me tremendous financial distress.',
       'When attempting to acquire my Experian credit report from XXXX I keep receiving this error message - " A condition exists that prevents Experian from being able to accept your request at this time. \n\nTo obtain your Experian XXXX XXXX XXXX, please mail your request to the address below using the XXXX XXXX XXXX  Request form. \'\' This is a violation because it is not free. If i have to mail in a request, i need to buy postage. Sending mail is not free and Experian has neglected to fix this issue for YEAR

In [64]:
tdf.complaint_what_happened.values[0:4]

array(['I have requested for the credit agency to remove/delete a bankruptcy that is inaccurate. I also requested for the company to provide detailed information on the process that is used to verify bankruptcies but the agency failed to provide this information. This has been very difficult to fix. It has been over 30 days with multiple disputes without a solution/explanation. The company has not deleted/ removed this from my report causing me tremendous financial distress.',
       'When attempting to acquire my Experian credit report from XXXX I keep receiving this error message - " A condition exists that prevents Experian from being able to accept your request at this time. \n\nTo obtain your Experian XXXX XXXX XXXX, please mail your request to the address below using the XXXX XXXX XXXX  Request form. \'\' This is a violation because it is not free. If i have to mail in a request, i need to buy postage. Sending mail is not free and Experian has neglected to fix this issue for YEAR